# Model Editing (ROME-style)

Surgically edit factual associations stored in model weights without
retraining. This notebook demonstrates ROME-style rank-one model editing:
locate the decisive MLP layer, compute edit vectors, and apply a
rank-one weight update.

This notebook covers the `irtk.model_editing` module.

## Why This Matters

ROME-style model editing modifies specific factual associations by making rank-one updates to MLP weights. Understanding how this works requires knowing where facts are stored and how MLP layers function as key-value memories.

**Key references:**
- [Meng et al. (2022) "Locating and Editing Factual Associations"](https://arxiv.org/abs/2202.05262) — ROME: rank-one model editing for fact modification
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import model_editing

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Locate the Decisive Layer

Which MLP layer stores the fact? We corrupt the subject token
and restore each layer's MLP output to find which one recovers
the prediction.

In [ ]:
prompt = "The Eiffel Tower is located in"
tokens = model.to_tokens(prompt)
token_strs = [tokenizer.decode([int(t)]) for t in tokens]

paris_id = tokenizer.encode(" Paris")[0]
def metric(logits):
    return float(logits[-1, paris_id])

# Find subject position (first token of 'Eiffel')
subject_pos = 1  # 'The' is 0, 'Eiffel' starts at 1
print(f"Subject token: '{token_strs[subject_pos]}'")

result = model_editing.locate_decisive_layer(
    model, tokens, metric, corrupt_pos=subject_pos, noise_std=3.0
)

print(f"\nClean metric:     {result['clean_metric']:.4f}")
print(f"Corrupted metric: {result['corrupted_metric']:.4f}")
print(f"Decisive layer:   {result['decisive_layer']}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(model.cfg.n_layers), result['layer_effects'], color='steelblue')
ax.axvline(result['decisive_layer'], color='red', linestyle='--', label=f"Decisive: L{result['decisive_layer']}")
ax.set_xlabel('Layer')
ax.set_ylabel('Metric recovery')
ax.set_title('MLP Layer Importance for Fact Recall')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Compute Edit Vectors

The **key vector** is what the MLP layer recognizes (the subject
representation). The **value vector** is what we want it to produce
(the target token direction).

In [ ]:
layer = result['decisive_layer']

# Key: what the MLP recognizes
key_vec = model_editing.compute_key_vector(model, tokens, layer=layer, pos=subject_pos)
print(f"Key vector norm: {np.linalg.norm(key_vec):.4f}")
print(f"Key vector shape: {key_vec.shape}")

# Value: desired output direction (London instead of Paris)
london_id = tokenizer.encode(" London")[0]
val_vec = model_editing.compute_value_target(
    model, tokens, layer=layer, pos=-1, target_token=london_id, coeff=5.0
)
print(f"Value vector norm: {np.linalg.norm(val_vec):.4f}")

## 3. Apply Rank-One Edit

Modify the MLP's output weights so it maps the key to the value.

In [ ]:
# Before edit
clean_logits = model(tokens)
clean_probs = jax.nn.softmax(clean_logits[-1])
print("Before edit:")
print(f"  P(' Paris'):  {float(clean_probs[paris_id]):.4f}")
print(f"  P(' London'): {float(clean_probs[london_id]):.4f}")

# Apply edit
edited = model_editing.apply_rank_one_edit(
    model, layer=layer, key_vector=key_vec, value_vector=val_vec
)

# After edit
edited_logits = edited(tokens)
edited_probs = jax.nn.softmax(edited_logits[-1])
print("\nAfter edit:")
print(f"  P(' Paris'):  {float(edited_probs[paris_id]):.4f}")
print(f"  P(' London'): {float(edited_probs[london_id]):.4f}")

# Show top predictions
top_k = 10
top_ids = jnp.argsort(edited_logits[-1])[::-1][:top_k]
print(f"\nTop {top_k} predictions after edit:")
for i, tid in enumerate(top_ids):
    tid = int(tid)
    print(f"  {i+1}. {tokenizer.decode([tid])!r}: {float(edited_probs[tid]):.4f}")

## 4. High-Level edit_fact API

All steps in one call: locate layer, compute vectors, apply edit.

In [ ]:
# Edit: "The Eiffel Tower is located in" -> " London"
edit_result = model_editing.edit_fact(
    model, tokens, subject_pos=subject_pos,
    target_token=london_id, coeff=5.0, metric_fn=metric
)

print(f"Edited layer: {edit_result['layer']}")
print(f"Key vector norm: {np.linalg.norm(edit_result['key_vector']):.4f}")
print(f"Value vector norm: {np.linalg.norm(edit_result['value_vector']):.4f}")

# Check the edit
edited_model = edit_result['edited_model']
e_logits = edited_model(tokens)
e_probs = jax.nn.softmax(e_logits[-1])
print(f"\nP(' Paris'):  {float(e_probs[paris_id]):.4f}")
print(f"P(' London'): {float(e_probs[london_id]):.4f}")

## 5. Specificity Check

Does the edit affect unrelated prompts? A good edit should be
specific to the targeted fact.

In [ ]:
# Test on related and unrelated prompts
test_prompts = [
    "The Eiffel Tower is located in",  # target (should change)
    "The capital of France is",          # related (may change)
    "The weather today is",              # unrelated (should not change)
    "My favorite food is",               # unrelated
]

edited_model = edit_result['edited_model']

print("Specificity analysis:")
for prompt in test_prompts:
    toks = model.to_tokens(prompt)
    clean_logits = model(toks)
    edited_logits = edited_model(toks)
    
    clean_top = int(jnp.argmax(clean_logits[-1]))
    edited_top = int(jnp.argmax(edited_logits[-1]))
    
    logit_diff = float(jnp.linalg.norm(edited_logits[-1] - clean_logits[-1]))
    
    print(f"\n  {prompt!r}")
    print(f"    Clean:  {tokenizer.decode([clean_top])!r}")
    print(f"    Edited: {tokenizer.decode([edited_top])!r}")
    print(f"    Logit L2 diff: {logit_diff:.4f}")

## Summary

| Function | Purpose |
|----------|--------|
| `locate_decisive_layer()` | Find which MLP layer stores a fact |
| `compute_key_vector()` | Extract subject representation at a layer |
| `compute_value_target()` | Compute desired MLP output delta |
| `apply_rank_one_edit()` | Apply rank-one update to MLP weights |
| `edit_fact()` | High-level: locate + compute + edit in one call |